In [ ]:
from types import resolve_bases
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import plotly.express as px
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from types import resolve_bases
import pickle
from sklearn.metrics import root_mean_squared_error

In [2]:
GrdSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A1_PGCI-GrdSrch-[27]-P3O1/raw-data_2023-03-10_PtA1-PGCI-GrdSrch-[27]-P3O1_Stykke-4.csv")
RndSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A2_PGCI-RndSrch-[27]-P3O1/raw-data_2023-03-10_PtA2-PGCI-RndSrch-[27]-P3O1_Stykke-4.csv")
GrdSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
RndSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B5_PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1/raw-data_2023-03-20_PtB5-PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1-Stykke-4.csv")
BOpt_8SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

In [3]:
df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df])
df.drop(columns=["mould_position","G_stoichiometry","CA_stoichiometry","IA_stoichiometry","StartPolymerMass_g","EndPolymerMass_pct"],inplace=True)
df['DeltaPolymerMass_pct']=df['DeltaPolymerMass_pct']*-1
df

,s1,s2,b1,DeltaPolymerMass_pct
0,0.8000,1.0000,0.8000,13.360997
1,1.0000,0.4000,0.6000,12.188133
2,0.4000,0.6000,0.8000,14.405027
3,0.8000,0.8000,1.0000,10.823033
4,0.4000,0.8000,0.6000,14.041169
...,...,...,...,...
24,0.4413,0.9805,0.5755,11.473868
25,0.4925,0.8819,0.5811,12.517877
26,0.3829,0.6579,0.7756,16.666833
27,0.3349,0.6460,0.7712,17.279822


In [4]:
X = df[["s1","s2","b1"]].to_numpy()
y = df["DeltaPolymerMass_pct"].to_numpy()

In [ ]:
# # Get 27 testing samples from only the grid and pseudorandom exercise
# X_train, X_test, y_train, y_test = train_test_split(X[0:54],y[0:54],test_size=0.5,random_state=7589)
# # Append all the rest of the samples to the training set
# y_train = np.append(y_train,y[54::])
# X_train = np.concatenate((X_train,X[54::]), axis=0)

In [ ]:
# with open('/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk2.pkl', 'rb') as file:
#     res = pickle.load(file)
# X_train = res["X_train"]
# X_test = res["X_test"]
# y_train = res["y_train"]
# y_test = res["y_test"]

In [ ]:
tree_methods = ['approx', 'exact', 'hist']

space = {
'tree_method': hp.choice('tree_method', tree_methods),
'max_depth': hp.randint('max_depth', 3, 12),
'n_estimators': hp.randint('n_estimators', 1, 1000),
'num_parallel_tree': hp.randint('num_parallel_tree', 2, 8),
'min_child_weight': hp.randint('min_child_weight', 1, 200),
'subsample': hp.uniform('subsample', 0.7, 1),
'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1),
'reg_lambda': hp.uniform('reg_lambda', 0, 10),
'learning_rate': hp.uniform('learning_rate', 0, 1),
}

RMSEs = []
r1 = 0
r2 = 10000
SeedRange = list(range(r1,r2))
seeds = []

def objective(params):
    xgb_model = xgb.XGBRegressor(**params)

    seed = np.random.choice(SeedRange)
    seeds.append(seed)
    X_train, X_test, y_train, y_test = train_test_split(X[0:54],y[0:54],test_size=0.5,random_state=seed)
    y_train = np.append(y_train,y[54::])
    X_train = np.concatenate((X_train,X[54::]), axis=0)

    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    RMSE = root_mean_squared_error(y_test,y_pred)
    RMSEs.append(RMSE)
    # RMSLE = np.sqrt(mean_squared_log_error(y_test, y_pred))
    # return {'loss': RMSLE, 'status': STATUS_OK}
    return {'loss': RMSE, 'status': STATUS_OK}

trials = Trials()

best_params = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials)
best_params["tree_method"] = tree_methods[best_params["tree_method"]]
print("Best set of hyperparameters: ", best_params)

In [6]:
# X_train, X_test, y_train, y_test = train_test_split(X[0:54],y[0:54],test_size=0.5,random_state=seeds[np.argmin(RMSEs)])
X_train, X_test, y_train, y_test = train_test_split(X[0:54],y[0:54],test_size=0.5,random_state=4347)
y_train = np.append(y_train,y[54::])
X_train = np.concatenate((X_train,X[54::]), axis=0)
# xgb_model = xgb.XGBRegressor(**best_params)
xgb_model = xgb.XGBRegressor(**{'colsample_bylevel': np.float64(0.6660789693068893), 'colsample_bytree': np.float64(0.8702659610797161), 'learning_rate': np.float64(0.0397937476235036), 'max_depth': np.int64(3), 'min_child_weight': np.int64(8), 'n_estimators': np.int64(439), 'num_parallel_tree': np.int64(3), 'reg_lambda': np.float64(9.983174869939603), 'subsample': np.float64(0.8680580739239264), 'tree_method': 'hist'})
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
RMSE = root_mean_squared_error(y_test, y_pred)
print("The score is %.5f" % RMSE )

The score is 0.73630


In [7]:
n = 10
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoords_lis.append([i,j,k])
ijkCoords_arr = np.array(ijkCoords_lis)
y_pred = xgb_model.predict(ijkCoords_lis)
df2 = pd.DataFrame({'s1': ijkCoords_arr[:, 0],'s2': ijkCoords_arr[:, 1],'b1': ijkCoords_arr[:, 2], 'y_pred': y_pred})
df2["y_pred"] = y_pred
fig = px.scatter_3d(df2, x='s1', y='s2', z='b1', color='y_pred')
fig.show()
print(np.max(y_pred))
print(ijkCoords_arr[np.argmax(y_pred)])

16.236893
[0.375 0.625 0.75 ]


In [8]:
n = 200
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoords_lis.append([i,j,k])
ijkCoords_arr = np.array(ijkCoords_lis)
y_pred = xgb_model.predict(ijkCoords_lis)
print(np.max(y_pred))
print(ijkCoords_arr[np.argmax(y_pred)])

17.501368
[0.33838384 0.64646465 0.77272727]


In [9]:
xgb_model.save_model('ModelMk1.json')

In [ ]:
# Obtained after 1000 epochs
# {'colsample_bylevel': np.float64(0.6660789693068893), 'colsample_bytree': np.float64(0.8702659610797161), 'learning_rate': np.float64(0.0397937476235036), 'max_depth': np.int64(3), 'min_child_weight': np.int64(8), 'n_estimators': np.int64(439), 'num_parallel_tree': np.int64(3), 'reg_lambda': np.float64(9.983174869939603), 'subsample': np.float64(0.8680580739239264), 'tree_method': 'hist'}
# np.int64(4347)
# RMSE = 0.73630